In [7]:
import os
import json
import time
import csv
import random
import torch

import numpy as np
import pandas as pd

from openai import OpenAI
from sentence_transformers import SentenceTransformer, util

In [8]:
DOMAINS = [
    "dsa", "pf", "oop", "os", "dbms", "cn",
    "bd", "fd", "sql", "sd", "cicd", "do",
    "ml", "da", "ba", "pm"
]

DIFFICULTY_LEVELS = [0, 1, 2, 3, 4]

In [9]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
api_key= user_secrets.get_secret("API_KEY")


In [10]:
client = OpenAI(api_key= api_key,base_url="https://openrouter.ai/api/v1")

MODEL_NAME = "xiaomi/mimo-v2-flash:free"

In [11]:
def call_llm(prompt, max_tokens=600, temperature=0.4):
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": "You are an expert computer science interviewer."},
            {"role": "user", "content": prompt}
        ],
        max_tokens=max_tokens,
        temperature=temperature
    )
    
    return response.choices[0].message.content


In [12]:
QUESTION_PROMPT = """
You are a controlled data generator for a computer science interview dataset.

Your task is to generate exactly {num_questions} interview questions.
You must follow EVERY rule below without exception.


GLOBAL CONSTRAINTS:

- You must output ONLY a valid JSON array.
- Do NOT include explanations, comments, markdown, or any text outside the JSON.
- Do NOT include trailing commas.
- Do NOT violate formatting rules even once.
- If any rule cannot be followed, generate a different question.


DIFFICULTY DEFINITION:

The difficulty level is an integer from 0 to 4, where:

- Difficulty 0 (easiest):
  Basic recall and definitions only.
  No comparisons, no scenarios, no trade-offs.

- Difficulty 1:
  Simple explanation of a single concept.
  Light reasoning allowed.
  Still focused on one idea only.

- Difficulty 2:
  Comparison-based or difference-based reasoning.
  Multiple related concepts allowed.
  Trade-offs and use-case discussion allowed.

- Difficulty 3:
  Scenario-based or case-based reasoning.
  Real-world situations, edge cases, or practical implications.
  Requires applied understanding.

- Difficulty 4 (hardest):
  System-level or design-oriented thinking.
  Focus on limitations, scalability, performance, or architectural decisions.
  May involve ambiguity or multiple valid approaches.

The "difficulty" field in every output object MUST be exactly {difficulty}.


DOMAIN CONSTRAINT (CRITICAL):


- ALL questions must belong STRICTLY to the domain: "{domain}".
- The question MUST be clearly and unambiguously related to this domain.
- Do NOT include concepts that primarily belong to other domains.
- Do NOT mix domains unless unavoidable, and even then keep focus on "{domain}".

The "domains" field MUST be:
["{domain}"]

Do NOT output any other domain values.


QUESTION FORMAT RULES:

Each item MUST be a JSON object with EXACTLY these fields:

1) "question"
2) "difficulty"
3) "domains"

Rules for "question":
- Must be a single sentence.
- Must NOT start with any numbering, labels, or prefixes.
- Must NOT contain a question mark (?).
- Must ALWAYS end with a full stop (.).
- Must be clear, concise, and technically precise.
- Must be suitable for a technical interview.

Rules for "difficulty":
- Must be the integer {difficulty}.

Rules for "domains":
- Must be exactly ["{domain}"].


QUESTION DESIGN REQUIREMENTS:


- Do NOT generate only definition-style questions.
- Use a mix of question types appropriate for the difficulty:
  - definition-based
  - explanation-based
  - comparison-based
  - scenario-based
  - reasoning or design-based

- Do NOT repeat or rephrase common textbook questions.
- Do NOT paraphrase previously generated questions.
- Each question must be conceptually distinct.


HALLUCINATION PREVENTION:


- Do NOT invent fictional technologies, APIs, tools, or frameworks.
- Do NOT use vague buzzwords without concrete meaning.
- Do NOT drift outside computer science topics.
- Do NOT include business, management, or non-technical content unless the domain explicitly requires it.


OUTPUT FORMAT:


Return ONLY a JSON array of objects.
No explanations.
No headings.
No additional text.

FINAL INSTRUCTION:

Generate exactly {num_questions} questions for:
- Domain: "{domain}"
- Difficulty: {difficulty}

Follow ALL rules strictly.
"""


In [13]:
embedder = SentenceTransformer("all-MiniLM-L6-v2")

accepted_questions = []
accepted_embeddings = []

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [14]:
def is_valid_question(item, expected_difficulty, expected_domain):
    if not isinstance(item, dict):
        return False

    if "question" not in item or "domains" not in item or "difficulty" not in item:
        return False

    text = item["question"].strip()

    if "?" in text:
        return False
    if not text.endswith("."):
        return False
    if item["difficulty"] != expected_difficulty:
        return False
    if item["domains"] != [expected_domain]:
        return False
    if len(text.split()) < 6:
        return False

    return True


In [15]:
def is_duplicate(question_text, threshold=0.9):

    if len(accepted_embeddings) == 0:
        return False


    new_emb = embedder.encode(question_text, convert_to_tensor=True)


    existing_embs = torch.stack(accepted_embeddings)

    similarities = util.cos_sim(new_emb, existing_embs)

    return similarities.max().item() > threshold


In [16]:
TARGET_PER_PAIR = 50   # example: 5 difficulties × 16 domains × 40 ≈ 3200
BATCH_SIZE = 10

results = []

for difficulty in DIFFICULTY_LEVELS:
    for domain in DOMAINS:
        print(f"\nGenerating for difficulty {difficulty}, domain {domain}")

        while len([
            r for r in results
            if r["difficulty"] == difficulty and r["domain"] == domain
        ]) < TARGET_PER_PAIR:

            prompt = QUESTION_PROMPT.format(
                num_questions=BATCH_SIZE,
                difficulty=difficulty,
                domain=domain
            )

            try:
                raw_output = call_llm(prompt)
                batch = json.loads(raw_output)
            except Exception:
                print("Model output invalid, retrying...")
                continue

            for item in batch:
                if not is_valid_question(item, difficulty, domain):
                    continue

                if is_duplicate(item["question"]):
                    continue

                emb = embedder.encode(item["question"], convert_to_tensor=True)

                accepted_questions.append(item["question"])
                accepted_embeddings.append(emb)

                results.append({
                    "question": item["question"],
                    "domain": domain,
                    "difficulty": difficulty
                })

            time.sleep(1)




df = pd.DataFrame(results)
df.to_csv("xiaomi_mimo_v2_2.csv", index=False, encoding="utf-8")

print("Saved", len(df), "questions.")



Generating for difficulty 0, domain dsa

Generating for difficulty 0, domain pf

Generating for difficulty 0, domain oop

Generating for difficulty 0, domain os

Generating for difficulty 0, domain dbms

Generating for difficulty 0, domain cn

Generating for difficulty 0, domain bd

Generating for difficulty 0, domain fd

Generating for difficulty 0, domain sql

Generating for difficulty 0, domain sd

Generating for difficulty 0, domain cicd

Generating for difficulty 0, domain do

Generating for difficulty 0, domain ml

Generating for difficulty 0, domain da

Generating for difficulty 0, domain ba

Generating for difficulty 0, domain pm

Generating for difficulty 1, domain dsa

Generating for difficulty 1, domain pf

Generating for difficulty 1, domain oop

Generating for difficulty 1, domain os

Generating for difficulty 1, domain dbms

Generating for difficulty 1, domain cn

Generating for difficulty 1, domain bd

Generating for difficulty 1, domain fd

Generating for difficulty 1,